[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_04/3_stats.ipynb)

# Day 4: Statistics, A/B testing and product (easy)

**How to work:** Answer in two layers: the number (code) and one plain sentence a product manager understands. Name the method and its assumptions before computing.

**Today (25 min):**

| # | Question | Level | Minutes |
|---|---|---|---|
| Q1 | How many of 20 new players come back? | easy | 7 |
| Q2 | Taxi pickups per hour | easy | 7 |
| Q3 | How heavy is a Gentoo penguin? | easy | 6 |
| Q4 | Which coin did I pick? (review) | easy | 5 |

**Handbook:** [Statistics, A/B testing and product](https://mahsa-agz.github.io/ds-handbook/#stats), [cheat sheets](https://mahsa-agz.github.io/ds-handbook/#cheat)

## Data

All real datasets (see DATASETS.md for sources). The setup cell loads them:

- `cookie`: Cookie Cats mobile game A/B test, 90,189 players. `version` is `gate_30` (control) or `gate_40` (treatment: the first gate moved from level 30 to 40); `sum_gamerounds` = rounds played in the first week; `retention_1`, `retention_7` = came back 1 / 7 days after install (bool).
- `taxis`: 6,433 New York taxi trips from March 2019 (`pickup`, `distance`, `fare`, `tip`, `total`, `payment`, `pickup_borough` ...).
- `penguins`: 344 penguins (`species`, `island`, `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g`, `sex`; some NaN).

Classic probability questions need no data: solve them on paper first, then check with a simulation (`rng = np.random.default_rng(0)`).

## Setup

Run this cell once before the questions.

In [ ]:
# Finds the data folder (local copy of the repository, or /content/data in Colab) and downloads a missing file
# from the ds-handbook repository on GitHub (or from its original source).
import os, sys, urllib.request

RAW = "https://raw.githubusercontent.com/mahsa-agz/mahsa-agz.github.io/main/ds-handbook/exam_prep/data/"

def _find_data():
    if "google.colab" in sys.modules:
        os.makedirs("/content/data", exist_ok=True)
        return "/content/data"
    for p in ("../data", "data", "exam_prep/data"):
        if os.path.isdir(p):
            return p
    os.makedirs("data", exist_ok=True)
    return "data"

DATA = _find_data()
print("data folder:", DATA)

def data_path(name, url=None):
    """Local path of a data file; downloads it from the repository (or from url) if it is not there yet."""
    path = os.path.join(DATA, name)
    if not os.path.exists(path):
        for source in (RAW + name, url):
            if not source:
                continue
            try:
                print("downloading", name)
                urllib.request.urlretrieve(source, path)
                break
            except Exception as e:
                print("  could not download from", source, "->", e)
    return path

import numpy as np
import pandas as pd
from scipy import stats as st
pd.set_option("display.width", 120)

cookie = pd.read_csv(data_path("cookie_cats.csv", "https://raw.githubusercontent.com/robguilarr/ab_testing_cookie_cats/main/datasets/cookie_cats.csv"))
taxis = pd.read_csv(data_path("taxis.csv", "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/taxis.csv"), parse_dates=["pickup", "dropoff"])
penguins = pd.read_csv(data_path("penguins.csv", "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"))
for _name in ['cookie', 'taxis', 'penguins']:
    print(_name, globals()[_name].shape)

---
### Q1. How many of 20 new players come back?

*easy, about 7 min*

In the control group of Cookie Cats (`version == 'gate_30'`), each player either comes back on day 1 (`retention_1`) or not.
1. Estimate `p`, the day-1 retention rate of the control group.
2. A community manager invites groups of 20 new players. Assuming players are independent, what is the probability that at least 10 of 20 come back? What are the expected count and its sd?
3. Check against the real data: shuffle the control players with `rng = np.random.default_rng(0)`, cut them into groups of 20, and print the share of groups with at least 10 returners, plus the mean and sd of the group counts.

**Then say it to a PM:** one or two plain sentences with the number and what it means for the product (no jargon).

In [ ]:
# cookie is loaded in the setup
ctrl = cookie[cookie.version == 'gate_30']

<details><summary><b>Hint 1</b></summary>

Signal: a fixed number of independent yes/no trials with the same probability. Topic: the binomial distribution `Binomial(n = 20, p)`.

</details>

<details><summary><b>Hint 2</b></summary>

1. `p = ctrl.retention_1.mean()`. 2. `st.binom.sf(9, 20, p)` is `P(X >= 10)` (sf(k) = P(X > k)). Mean `n p`, sd `sqrt(n p (1 - p))`. 3. `r = rng.permutation(ctrl.retention_1.to_numpy())`, cut to a multiple of 20, `reshape(-1, 20).sum(axis=1)`.

</details>

<details><summary><b>Solution</b></summary>

```python
ctrl = cookie[cookie["version"] == "gate_30"]
p = ctrl["retention_1"].mean()
n = 20
print(f"p = {p:.4f}")
print(f"model: P(X >= 10) = {st.binom.sf(9, n, p):.3f}   mean {n * p:.2f}   sd {np.sqrt(n * p * (1 - p)):.2f}")

rng = np.random.default_rng(0)
r = rng.permutation(ctrl["retention_1"].to_numpy().astype(int))
groups = r[: len(r) // n * n].reshape(-1, n).sum(axis=1)
print(f"data:  P(X >= 10) = {(groups >= 10).mean():.3f}   mean {groups.mean():.2f}   sd {groups.std(ddof=1):.2f}"
      f"   ({len(groups)} groups)")

# Output:
# p = 0.4482
# model: P(X >= 10) = 0.402   mean 8.96   sd 2.22
# data:  P(X >= 10) = 0.403   mean 8.96   sd 2.20   (2235 groups)
```

**Why:** Each player is a Bernoulli trial with the same `p`, and a random shuffle makes the players in a group independent, so the count of returners is `Binomial(20, 0.448)`. The model and the data agree on the tail probability, the mean (`n p` = 8.96) and the sd (`sqrt(n p (1 - p))`, about 2.22). `st.binom.sf(9, ...)` gives `P(X > 9) = P(X >= 10)`: an off-by-one here is the most common bug. When would the binomial fail? If the 20 players are friends who joined together, their behaviour is correlated and the real sd is larger than the binomial sd (overdispersion).

**Say it to a PM:** About 45% of new players come back the next day, so in a group of 20 we expect about 9, and a group reaching 10 or more happens roughly 4 times in 10. A group of 20 with 4 or fewer returners happens only about 2% of the time, so it would be worth a look.

**Common mistakes:** `st.binom.sf(10, ...)` (that is `P(X >= 11)`). Using the normal approximation without a continuity correction for such a small `n`. Assuming independence for players who joined together.

**Learn more:** [stats-distributions](https://mahsa-agz.github.io/ds-handbook/#stats-distributions)

</details>

---
### Q2. Taxi pickups per hour

*easy, about 7 min*

A dispatcher models the number of taxi pickups per hour as Poisson. Using `taxis` (keep March 2019 only):
1. Count pickups in every hour of March (include hours with 0 pickups). Print the mean and the variance of the hourly counts.
2. Under a Poisson model with that mean, what share of hours would have 0 pickups? What share do we see?
3. Is Poisson a good model here? Why or why not?

Reminder: a Poisson count has variance equal to its mean.

**Then say it to a PM:** one or two plain sentences with the number and what it means for the product (no jargon).

In [ ]:
# taxis is loaded in the setup (pickup is a datetime)

<details><summary><b>Hint 1</b></summary>

Signal: counts of events in fixed time windows. Topic: the Poisson distribution and its check `variance = mean` (dispersion).

</details>

<details><summary><b>Hint 2</b></summary>

1. `m = taxis[taxis.pickup >= '2019-03-01']`; `h = m.set_index('pickup').resample('h').size()` (resample includes empty hours). 2. `h.mean()`, `h.var()`. 3. `st.poisson.pmf(0, h.mean())` versus `(h == 0).mean()`. 4. Also look at mean and variance inside one hour of the day.

</details>

<details><summary><b>Solution</b></summary>

```python
m = taxis[taxis["pickup"] >= "2019-03-01"]
h = m.set_index("pickup").resample("h").size()
lam = h.mean()
print(f"hours {len(h)}   mean {lam:.2f}   variance {h.var():.2f}   ratio {h.var() / lam:.2f}")
print(f"P(0 pickups): Poisson {st.poisson.pmf(0, lam):.5f}   observed {(h == 0).mean():.4f}")
by_hour = h.groupby(h.index.hour).agg(["mean", "var"])
print(f"inside one hour of the day, mean of var/mean ratios: {(by_hour['var'] / by_hour['mean']).mean():.2f}")

# Output:
# hours 744   mean 8.65   variance 26.37   ratio 3.05
# P(0 pickups): Poisson 0.00018   observed 0.0457
# inside one hour of the day, mean of var/mean ratios: 1.63
```

**Why:** A Poisson model assumes a constant rate. Taxi demand has a strong daily cycle (about 2 pickups an hour at 4 am, 13 at 7 pm in this sample), so the hourly counts are a mix of Poisson counts with different rates. A mix always has variance above its mean (here about 3 times the mean), and many more empty hours than a single Poisson predicts. Even inside one hour of the day the ratio stays above 1 (about 1.6 on average), because weekdays and weekends differ. Fixes: model the rate by hour and weekday, or use a negative binomial, which has an extra parameter for overdispersion.

**Say it to a PM:** Demand is far less steady than a simple model assumes: about 5% of hours had no pickups in this sample, while the simple model says that should almost never happen. Staffing plans need to use the rate for each hour and weekday, not one average.

**Common mistakes:** Counting with `groupby(dt.hour)` and missing the empty hours (zeros never appear in a groupby). Accepting Poisson without checking `variance = mean`. Forgetting that this file is a sample of all trips, so the absolute counts are small.

**Learn more:** [stats-distributions](https://mahsa-agz.github.io/ds-handbook/#stats-distributions), [stats-descriptive](https://mahsa-agz.github.io/ds-handbook/#stats-descriptive)

</details>

---
### Q3. How heavy is a Gentoo penguin?

*easy, about 6 min*

Using `penguins`, keep species Gentoo (drop missing `body_mass_g`).
1. Print the mean and sd of body mass.
2. Treat body mass as normal with that mean and sd. What share of Gentoo should weigh more than 6,000 g? What share actually does?
3. Check the 68-95 rule: what share is within 1 sd and within 2 sd of the mean?
4. A Gentoo weighs 6,300 g. What is its z-score?

**Then say it to a PM:** one or two plain sentences with the number and what it means for the product (no jargon).

In [ ]:
# penguins is loaded in the setup

<details><summary><b>Hint 1</b></summary>

Signal: a continuous measurement, symmetric, from one homogeneous group. Topic: the normal distribution, z-scores and the 68-95-99.7 rule.

</details>

<details><summary><b>Hint 2</b></summary>

1. `g = penguins.loc[penguins.species == 'Gentoo', 'body_mass_g'].dropna()`. 2. `st.norm.sf(6000, mu, sd)` versus `(g > 6000).mean()`. 3. `(abs(g - mu) < sd).mean()`. 4. `z = (6300 - mu) / sd`.

</details>

<details><summary><b>Solution</b></summary>

```python
g = penguins.loc[penguins["species"] == "Gentoo", "body_mass_g"].dropna()
mu, sd = g.mean(), g.std()
print(f"n {len(g)}   mean {mu:.0f} g   sd {sd:.0f} g")
print(f"P(mass > 6000): normal {st.norm.sf(6000, mu, sd):.3f}   observed {(g > 6000).mean():.3f} ({(g > 6000).sum()} birds)")
print(f"within 1 sd {(abs(g - mu) < sd).mean():.3f}   within 2 sd {(abs(g - mu) < 2 * sd).mean():.3f}")
print(f"z-score of 6300 g: {(6300 - mu) / sd:.2f}")

# Output:
# n 123   mean 5076 g   sd 504 g
# P(mass > 6000): normal 0.033   observed 0.016 (2 birds)
# within 1 sd 0.675   within 2 sd 0.984
# z-score of 6300 g: 2.43
```

**Why:** The 68-95 rule fits well (about 67% and 98%), so the normal model describes the middle of the data. The far tail is where it is weakest: the model says about 3% are above 6,000 g, the data has 2 birds out of 123 (1.6%). With 123 birds, a tail share is estimated from only a handful of points, so neither number is precise. Note that males and females differ, so the full mass distribution is a mix of two normals; a model by sex would fit better. A z-score of 2.43 means 6,300 g is 2.43 sd above the mean, unusual but not impossible (about 1 in 130 under the model).

**Say it to a PM:** A typical Gentoo weighs about 5,080 g, and two thirds weigh within about 500 g of that. A 6,300 g bird is among the heaviest we would expect to see, roughly 1 in 130.

**Common mistakes:** Using the normal model for far tails or for skewed data (money, counts). Using the population sd formula by accident (`np.std` divides by n). Pooling males and females and calling the mix normal.

**Learn more:** [stats-distributions](https://mahsa-agz.github.io/ds-handbook/#stats-distributions)

</details>

---
### Q4. Which coin did I pick? (review)

*easy, about 5 min*

Review of Bayes. A bag has 2 coins: one fair and one with heads on both sides. You pick one at random and flip it 3 times: heads, heads, heads.
1. What is the probability you picked the two-headed coin?
2. What is the probability that the next flip is heads?
3. Check part 1 with a simulation of 200,000 picks (`rng = np.random.default_rng(0)`).

**Then say it to a PM:** one or two plain sentences with the number and what it means for the product (no jargon).

In [ ]:
# your code here

<details><summary><b>Hint 1</b></summary>

Signal: 'given what you observed, which hidden case is it?'. Topic: Bayes' rule with two hypotheses, then the law of total probability for the next flip.

</details>

<details><summary><b>Hint 2</b></summary>

1. Prior 1/2 each. Likelihood of HHH: fair `(1/2)^3 = 1/8`, two-headed 1. 2. Posterior = `1 * 1/2 / (1 * 1/2 + 1/8 * 1/2)`. 3. Next heads = `post * 1 + (1 - post) * 1/2`. 4. Simulation: draw the coin, draw 3 flips, keep only the runs with 3 heads.

</details>

<details><summary><b>Solution</b></summary>

```python
post = (1 * 0.5) / (1 * 0.5 + (1 / 8) * 0.5)
print(f"P(two-headed | HHH) = {post:.4f}   P(next is heads) = {post + (1 - post) * 0.5:.4f}")

rng = np.random.default_rng(0)
n = 200_000
two_headed = rng.random(n) < 0.5
p_heads = np.where(two_headed, 1.0, 0.5)
hhh = (rng.random((n, 3)) < p_heads[:, None]).all(axis=1)
print(f"simulated P(two-headed | HHH) = {two_headed[hhh].mean():.4f}   (runs kept: {hhh.sum()})")

# Output:
# P(two-headed | HHH) = 0.8889   P(next is heads) = 0.9444
# simulated P(two-headed | HHH) = 0.8909   (runs kept: 112638)
```

**Why:** The evidence HHH is 8 times more likely under the two-headed coin (likelihood ratio 1 / (1/8) = 8). The prior odds are 1 : 1, so the posterior odds are 8 : 1, which is 8/9 = 0.889. The next flip mixes both coins with the posterior weights: `8/9 x 1 + 1/9 x 1/2 = 17/18 = 0.944`. The simulation keeps only the runs that match the evidence, which is exactly what conditioning means.

**Say it to a PM:** After three heads in a row we are about 89% sure it is the trick coin, and the next flip is heads with about 94% probability. Evidence moves our belief, but the starting belief still matters.

**Common mistakes:** Answering 1/2 (ignoring the evidence) or 1 (ignoring the fair coin). Saying the next flip is 1/2 (the coin is uncertain, so we average over both).

**Learn more:** [stats-bayes](https://mahsa-agz.github.io/ds-handbook/#stats-bayes), [stats-probability-puzzles](https://mahsa-agz.github.io/ds-handbook/#stats-probability-puzzles)

</details>

---
## Redo list

Write down every question you got wrong, solved only after Hint 2, or finished over time. Solve it again from scratch 3 days and 7 days later.

**Next:** [AI and ML](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_04/4_ai.ipynb)